In [3]:
import pandas as pd
import joblib

def predict_new_csv(
    model_path="rf_pipeline2900.joblib",
    new_csv="pred2900.csv",
    sep=",",
    output_csv="cool_results2900.csv"
):
    model = joblib.load(model_path)

    df = pd.read_csv(new_csv, sep=sep)

    feature_cols = ["wave", "intensity", "brain_region"]
    X_new = df[feature_cols].copy()

    pred = model.predict(X_new)
    proba = model.predict_proba(X_new)

    result = df.copy()
    result["predicted_class"] = pred

    class_names = model.named_steps["classifier"].classes_
    for i, cls in enumerate(class_names):
        result[f"proba_{cls}"] = proba[:, i]

    result.to_csv(output_csv, index=False, sep=sep)
    return result


if __name__ == "__main__":
    result = predict_new_csv(
        model_path="rf_pipeline2900.joblib",
        new_csv="pred2900.csv",
        output_csv="cool_results2900.csv"
    )

    print(result.head())


          wave     intensity brain_region predicted_class  proba_control  \
0  3069.991211  12840.918945     striatum             exo            0.0   
1  3069.208008  12492.286133     striatum             exo            0.0   
2  3068.425781  12823.419922     striatum             exo            0.0   
3  3067.642578  12516.583984     striatum             exo            0.0   
4  3066.859375  12546.350586     striatum             exo            0.0   

   proba_endo  proba_exo  
0         0.0        1.0  
1         0.0        1.0  
2         0.0        1.0  
3         0.0        1.0  
4         0.0        1.0  


In [5]:
import joblib
import pandas as pd

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


def test_model_on_new_csv(
    model_path="rf_2900_pipeline.joblib",
    test_csv="pred.csv",
    feature_cols=("wave", "intensity", "brain_region"),
    target_col="class",
    sep=",",
    output_csv="new_test_predictions.csv",
    save_only_wrong=False
):
    model = joblib.load(model_path)
    df = pd.read_csv(test_csv, sep=sep)

    missing_cols = [col for col in feature_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"В новом CSV нет колонок: {missing_cols}")

    X_test = df[list(feature_cols)].copy()

    y_pred = model.predict(X_test)

    result = df.copy()
    result["predicted_class"] = y_pred

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        classes = model.named_steps["classifier"].classes_
        for i, cls in enumerate(classes):
            result[f"proba_{cls}"] = proba[:, i]

    metrics = None

    if target_col in df.columns:
        y_true = df[target_col].copy()

        acc = accuracy_score(y_true, y_pred)
        report = classification_report(y_true, y_pred, zero_division=0)
        cm = confusion_matrix(y_true, y_pred)

        result["is_correct"] = (y_true == y_pred)

        metrics = {
            "accuracy": acc,
            "classification_report": report,
            "confusion_matrix": cm
        }

    if save_only_wrong and target_col in result.columns and "is_correct" in result.columns:
        result_to_save = result[result["is_correct"] == False].copy()
    else:
        result_to_save = result

    result_to_save.to_csv(output_csv, index=False, sep=sep)

    return result, metrics


if __name__ == "__main__":
    result_df, metrics = test_model_on_new_csv(
        model_path="rf_2900_pipeline.joblib",
        test_csv="pred.csv",
        feature_cols=("wave", "intensity", "brain_region"),
        target_col="class",
        output_csv="another_file_predictions.csv",
        save_only_wrong=False
    )

    print("Первые строки результата:")
    print(result_df.head())

    if metrics is not None:
        print("\nAccuracy:", metrics["accuracy"])
        print("\nClassification report:\n", metrics["classification_report"])
        print("\nConfusion matrix:\n", metrics["confusion_matrix"])
    else:
        print("\nВ новом CSV нет колонки 'class', поэтому метрики качества не считались.")


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done  80 out of  80 | elapsed:    0.1s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done  80 out of  80 | elapsed:    0.2s finished


Первые строки результата:
          wave     intensity brain_region predicted_class  proba_control  \
0  1699.966797  11496.750000   cerebellum             exo       0.037013   
1  1698.950195  11356.371094   cerebellum             exo       0.037013   
2  1697.932617  11349.275391   cerebellum             exo       0.037013   
3  1696.915039  11440.852539   cerebellum             exo       0.037013   
4  1695.897461  11384.416016   cerebellum             exo       0.037013   

   proba_endo  proba_exo  
0    0.000485   0.962501  
1    0.000485   0.962501  
2    0.000485   0.962501  
3    0.000485   0.962501  
4    0.000485   0.962501  

В новом CSV нет колонки 'class', поэтому метрики качества не считались.
